# Scenario 2: Code Generation with Claude Code

## 1. In One Minute

**What it is.** A team uses Claude Code to write and change code. It relies on custom slash commands, `CLAUDE.md` files, path-scoped rules, and a choice between plan mode and direct execution.

**Domains hit:** D3 Claude Code Configuration and Workflows, D5 Context Management.

**Task statements:** TS 3.1 (CLAUDE.md hierarchy), 3.2 (slash commands and skills), 3.3 (path-specific rules), 3.4 (plan mode vs direct), 3.5 (iterative refinement), 5.1 (context).

**Top 3 traps**
1. Putting shared team config in user-level files (`~/.claude/`) that teammates never receive.
2. Using plan mode for a tiny fix, or direct execution for a large architectural change.
3. Stuffing everything into one huge `CLAUDE.md` instead of using path rules and skills.

## 2. Architecture Diagram

```mermaid
---
config:
  htmlLabels: false
---
flowchart TD
    Dev["Developer request"] --> Mode{"Scope of change?"}
    Mode -->|"Small, clear, one file"| Direct["Direct execution"]
    Mode -->|"Multi-file, architectural,<br>several valid designs"| Plan["Plan mode<br>(read-only exploration)"]
    Plan --> Review["Human approves plan"]
    Review --> Direct
    Direct --> Ctx["Context loaded"]
    Ctx --> Root["Root CLAUDE.md<br>(universal facts, short)"]
    Ctx --> Rules[".claude/rules/*.md<br>(loaded by path glob)"]
    Ctx --> Cmd[".claude/commands/<br>(shared slash commands)"]
    Ctx --> Skill[".claude/skills/<br>(procedures, optional context: fork)"]
```

## 3. How a Good Answer Thinks

1. **Who needs it?** Whole team in the repo means project scope (`.claude/`, committed). Just me means user scope (`~/.claude/`).
2. **When should it load?** Always: `CLAUDE.md`. Only for some files: `.claude/rules/` with a path glob. Only on demand: a skill or command.
3. **How big is the change?** Large, multi-file, or unclear design: plan mode first. Small and clear: direct.
4. **Is the request vague?** Ask questions (interview) before writing code. Do not guess a design.
5. **Does a task produce lots of noisy output?** Run it in a forked context (`context: fork`) so the main session stays clean.
6. **Root cause first.** If Claude ignores a convention, check where the convention lives and when it loads.

### Model and Sampling Notes
- Use `claude-sonnet-4-5` for coordinators and reasoning, `claude-haiku-4-5` for workers and extraction.
- With extended thinking on: omit `temperature`, leave `top_k` and `top_p` unset, and use only `tool_choice` `auto` or `none` (forced or `any` returns HTTP 400). Forced-tool extraction calls must not use thinking.
- In tool loops, return thinking blocks with their signature unchanged.
- Loops end on `stop_reason` only (`end_turn`, `tool_use`, `max_tokens`, `stop_sequence`, `pause_turn`, `refusal`). A max-iterations counter is a safety cap, not the stop rule.

## 4. Task Statement Map

| Task statement | How it shows up here | Trap |
|---|---|---|
| TS 3.1 CLAUDE.md | Short root file with universal facts; keep it under about 200 lines | One giant file with every procedure |
| TS 3.2 Commands and skills | `/review` in `.claude/commands/`; verbose workflows in skills with `context: fork` | Putting shared commands in `~/.claude/commands/` |
| TS 3.3 Path rules | `.claude/rules/*.md` with `paths:` globs such as `backend/**` | Repeating the same rule in many folder-level files |
| TS 3.4 Plan mode | Plan mode for a multi-service refactor; direct for a typo | Always plan, or never plan |
| TS 3.5 Iterative refinement | Interview first when auth or database choice is not stated | Picking a design silently |
| TS 5.1 Context | Load rules by path; fork noisy work | Loading everything into every session |

## 5. Worked Example

**Request:** "Add authentication to the backend and update the API clients."

1. Many files and no stated auth method, so this is plan mode plus an interview.
2. Claude asks: JWT bearer tokens or session cookies? It waits for the answer.
3. In plan mode it reads the code, lists affected files, and proposes phases. Nothing is edited yet.
4. The human approves the plan.
5. Direct execution starts. Editing `backend/**` loads the backend rule file (async handlers, validated models). Frontend rules stay unloaded.
6. A shared `/review` command from `.claude/commands/` is used before the PR.

Try it offline: `scenarios/scenario_2_code_generation/simulation.py` (`ClaudeCodeGenerationEngine.evaluate_task`).

## 7. Self-Check Checklist

- [ ] I can state project scope vs user scope for commands and config.
- [ ] I know when rules load by glob versus always.
- [ ] I can justify plan mode vs direct execution in one sentence.
- [ ] I know what `context: fork` does and why it helps.
- [ ] I ask first when the request is vague.

**Memory hook:** "Share in the repo, load by path, plan big, fork the noise."


In [ ]:
# Step 1: Environment Setup & Simulation Import
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scenarios.scenario_2_code_generation.simulation import *

print(' Scenario Workbench Initialized: Scenario 2: Code Generation & Architectural Refactoring')
print(' Zero live API calls, 100% offline determinism')


### 🧪 Step 2: Core Execution Flow & Happy Path Simulation
Demonstrating end-to-end pipeline execution with audit logging.


In [ ]:
# Test Execution Mode Routing
engine = ClaudeCodeGenerationEngine()

# Test 1: Single file fix -> Direct Execution
task_single = CodeGenTask(
    task_id="t1",
    description="Fix typo in Button component",
    target_files=["frontend/ui/Button.tsx"],
    is_architectural_shift=False
)
outcome_single = engine.evaluate_task(task_single)
print("Single-file Mode:", outcome_single.mode_selected)
assert outcome_single.mode_selected == ExecutionMode.DIRECT_EXECUTION
assert "frontend_react" in outcome_single.injected_path_rules

# Test 2: Multi-file refactor -> Plan Mode
task_multi = CodeGenTask(
    task_id="t2",
    description="Refactor user models and controllers",
    target_files=["backend/models.py", "backend/controllers.py"],
    is_architectural_shift=True
)
outcome_multi = engine.evaluate_task(task_multi)
print("Multi-file Mode:", outcome_multi.mode_selected)
assert outcome_multi.mode_selected == ExecutionMode.PLAN_MODE
assert outcome_multi.forked_execution_used is True



### ⚠️ Step 3: Edge Case Testing & Guardrail Verifications
Testing boundary conditions, safety gates, error recovery, and escalation protocols.


In [ ]:
# Test Interview Pattern on Ambiguous Tech Stack
task_amb = CodeGenTask(
    task_id="t3",
    description="Implement user auth service",
    target_files=["backend/api/auth.py"],
    is_architectural_shift=False
)
outcome_amb = engine.evaluate_task(task_amb)
print("Interview Required:", outcome_amb.interview_required)
print("Clarifying Options:", outcome_amb.clarification_options)
assert outcome_amb.interview_required is True
assert len(outcome_amb.clarification_options) >= 2



### ✅ Step 4: Verification Summary & SLA Compliance
All assertions passed with zero unauthorized actions and 100% deterministic compliance.
